# Vision Language Models for Semantic-Aware Radiology Report Generation

## Motivation

Radiology report generation is an important application of vision-language models in healthcare. The goal is to automatically generate clinically meaningful reports from chest X-ray images. This can help reduce reporting time, support radiologists, and improve accessibility in medical systems with limited resources.

Most existing approaches focus heavily on improving traditional evaluation metrics such as BLEU and ROUGE. However, in medical text generation, exact wording is not always the best indicator of correctness. Two reports can describe the same clinical condition using different terminology while receiving a low lexical score.

For example:

- "cardiomegaly observed"
- "enlarged heart detected"

These statements are semantically similar, but lexical metrics may still penalize them because the wording differs.

Another major issue in radiology report generation is hallucination. Models may generate findings that are not visible in the image or repeat medically unsupported statements. This creates a serious challenge because fluent language does not always mean clinically correct language.

Because of this, semantic correctness and clinical consistency become more important than exact word overlap alone.

In this project, the focus shifts from only improving lexical metrics toward understanding and evaluating semantic alignment in generated reports. Alongside standard evaluation metrics, we explore:

- semantic-aware evaluation
- clinically-aware scoring
- hallucination analysis
- latent feature interpretation
- attention-based grounding analysis

The objective is not only to generate fluent reports, but also to better understand whether the generated reports preserve meaningful clinical information.

## 1. Environment Setup

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader
from torchvision import models

warnings.filterwarnings("ignore")

#### Reproducibility Setup

Deep learning experiments can produce slightly different results across runs because of random initialization, shuffling, and GPU operations.

To improve reproducibility, fixed random seeds are used across:
- Python
- NumPy
- PyTorch

This helps make experiments more stable and easier to compare during ablation studies and evaluation.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed set to: {SEED}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())

### Project Pipeline Overview

The overall pipeline used in this project consists of:

1. Chest X-ray preprocessing
2. Report text preprocessing
3. CNN-based image feature extraction
4. Attention-based report generation
5. Semantic-aware evaluation
6. Interpretability and hallucination analysis

The baseline architecture uses:
- ResNet18 as the visual encoder
- Bahdanau attention
- GRU decoder

Additional improvements explored include:
- beam search decoding
- coverage loss
- staged fine-tuning
- semantic-aware evaluation metrics
- clinically-aware scoring methods

## 2. Dataset Preparation



#### Indiana University Chest X-ray Dataset

This project uses the Indiana University Chest X-ray dataset, which is a commonly used benchmark for radiology report generation research.

The dataset contains:
- frontal and lateral chest X-ray images
- paired radiology reports
- findings and impression sections
- study-level identifiers

Each sample consists of:
1. a chest X-ray image
2. a corresponding radiology report

Unlike standard image classification datasets, this task requires the model to learn both:
- visual understanding
- clinically meaningful language generation

The dataset is relatively small compared to large-scale medical datasets such as MIMIC-CXR. Because of this, careful preprocessing and data splitting become especially important to avoid data leakage and overfitting.

In [ ]:
IMAGE_DIR = "/content/images/images_normalized/"
REPORT_PATH = "/content/indiana_reports.csv"

print("Image Directory:", IMAGE_DIR)
print("Report File:", REPORT_PATH)

### 2.1 Dataset Overview

Before training the model, it is important to understand:
- dataset size
- report structure
- vocabulary distribution
- image-report pairing

This helps identify:
- missing values
- duplicated studies
- imbalance issues
- preprocessing requirements

In [ ]:
df = pd.read_csv(REPORT_PATH)

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
print("Number of Samples:", len(df))

print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

### Paired Image-Report Structure

Each row in the dataset contains:
- an image filename
- a corresponding radiology report

The model learns to map:
- visual chest X-ray features
to
- clinically meaningful textual descriptions

This paired structure forms the foundation of the vision-language learning pipeline.

sample = df.iloc[0]

print("Image Path:")
print(sample['image_path'])

print("\nReport:")
print(sample['report'])

### Report Length Analysis

Radiology reports vary in length depending on:
- disease complexity
- number of findings
- reporting style

Understanding report length distribution helps determine:
- maximum sequence length
- padding strategy
- decoder efficiency

In [ ]:
df['report_length'] = df['report'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8,5))
plt.hist(df['report_length'], bins=30)

plt.xlabel("Report Length")
plt.ylabel("Frequency")
plt.title("Distribution of Report Lengths")

plt.show()

### 2.2 UID-Level Splitting

One major issue in medical imaging research is data leakage.

If images from the same study appear in both training and testing sets, the model may memorize patterns rather than genuinely generalize.

To avoid this, splitting is performed at the study or UID level instead of random image-level splitting.

This ensures:
- cleaner evaluation
- better generalization testing
- more reliable performance estimates

In [ ]:
from sklearn.model_selection import train_test_split

unique_uids = df['uid'].unique()

train_uids, temp_uids = train_test_split(
    unique_uids,
    test_size=0.2,
    random_state=SEED
)

val_uids, test_uids = train_test_split(
    temp_uids,
    test_size=0.5,
    random_state=SEED
)

train_df = df[df['uid'].isin(train_uids)]
val_df = df[df['uid'].isin(val_uids)]
test_df = df[df['uid'].isin(test_uids)]

In [ ]:
# split stats

print("Training Samples:", len(train_df))
print("Validation Samples:", len(val_df))
print("Testing Samples:", len(test_df))

print("\nUnique Training UIDs:", train_df['uid'].nunique())
print("Unique Validation UIDs:", val_df['uid'].nunique())
print("Unique Testing UIDs:", test_df['uid'].nunique())

### Why UID-Level Splitting Matters

This step is especially important in radiology datasets because:
- multiple images may belong to the same patient or study
- duplicated patterns can artificially inflate metrics
- leakage creates unrealistic evaluation performance

UID-level splitting helps ensure that the model is evaluated on genuinely unseen studies.

### 2.3 Image Loading and Preprocessing

Medical images require careful preprocessing before being passed into the encoder network.

The preprocessing pipeline includes:
- image resizing
- tensor conversion
- normalization

Images are resized to a fixed resolution to ensure consistent input dimensions for the CNN encoder.

In [ ]:
IMAGE_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# normalisation 

from PIL import Image

sample_image_path = os.path.join(
    IMAGE_DIR,
    train_df.iloc[0]['image_path']
)

image = Image.open(sample_image_path).convert("RGB")

plt.figure(figsize=(5,5))
plt.imshow(image)
plt.title("Sample Chest X-ray")
plt.axis("off")

plt.show()

In [ ]:
image_tensor = transform(image)

print("Tensor Shape:", image_tensor.shape)

- Normalization helps stabilize training by scaling pixel intensities into a more consistent range.

- The ImageNet normalization statistics are used because the encoder backbone is initialized using pretrained ImageNet weights.

### Final Preprocessing Pipeline

At this stage:
- reports have been paired with images
- UID-level splitting has been completed
- image preprocessing has been defined
- transformed tensors are ready for model training

The next stage focuses on:
- text preprocessing
- vocabulary construction
- tokenization
- sequence preparation
for the decoder network.

## 3. Exploratory Data Analysis (EDA)

Before training the model, it is important to understand the dataset beyond just loading images and reports into memory.

In medical vision-language tasks, dataset structure can strongly influence:
- language generation quality
- vocabulary learning
- semantic consistency
- class imbalance
- hallucination behavior

This section explores:
- report length distribution
- vocabulary characteristics
- common clinical terminology
- imbalance between normal and abnormal studies
- sample image-report pairs

Understanding these patterns helps guide later decisions involving:
- tokenizer design
- sequence length selection
- training strategy
- evaluation methodology

### 3.1 Report Length Distribution

Radiology reports can vary significantly in length depending on:
- the number of findings
- disease complexity
- reporting style

Very short reports may contain limited clinical detail, while longer reports often describe multiple findings and impressions.

Understanding the report length distribution helps determine:
- maximum decoder sequence length
- padding requirements
- truncation strategy
- memory usage during training

In [ ]:
# report length distribution

df['report_length'] = df['report'].apply(
    lambda x: len(str(x).split())
)

print(df['report_length'].describe())

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df['report_length'], bins=30)

plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.title("Distribution of Radiology Report Lengths")

plt.show()

### Observations

Most reports fall within a moderate word range, with a smaller number of longer reports containing more detailed findings.

This suggests that:
- a fixed maximum sequence length can reasonably cover most samples
- extremely long reports are relatively uncommon
- decoder efficiency can be improved without losing significant information

The distribution also highlights the variability of clinical language in radiology reporting.

### 3.2 Vocabulary Analysis

Radiology report generation depends heavily on learning clinically meaningful language patterns.

Before building the tokenizer and vocabulary, it is useful to examine:
- total vocabulary size
- frequency distribution
- repeated medical terminology
- common descriptive phrases

This gives a better understanding of the linguistic complexity of the dataset.

In [ ]:
from collections import Counter

all_words = []

for report in df['report']:
    words = str(report).lower().split()
    all_words.extend(words)

word_counter = Counter(all_words)

vocab_size = len(word_counter)

print("Vocabulary Size:", vocab_size)

In [ ]:
# most common words

top_words = word_counter.most_common(20)

top_words_df = pd.DataFrame(
    top_words,
    columns=['Word', 'Frequency']
)

top_words_df

### Observations

The vocabulary contains a mixture of:
- anatomical terms
- disease descriptions
- positional terminology
- reporting phrases

Some words appear extremely frequently because radiology reports often follow semi-structured clinical language patterns.

This repetitive structure can help the model learn medical phrasing more effectively, but it can also increase the risk of:
- repetitive generation
- template-style outputs
- hallucinated default findings

### 3.3 Top Clinical Terms

Beyond general vocabulary statistics, it is important to specifically examine clinically relevant terminology.

Frequent clinical terms provide insight into:
- common pathologies in the dataset
- reporting trends
- class imbalance
- dominant medical concepts learned by the model

In [ ]:
clinical_terms = [
    'cardiomegaly',
    'opacity',
    'effusion',
    'pneumonia',
    'edema',
    'atelectasis',
    'nodule',
    'normal',
    'infiltrate',
    'consolidation'
]

clinical_counts = {}

for term in clinical_terms:
    clinical_counts[term] = sum(
        term in str(report).lower()
        for report in df['report']
    )

clinical_df = pd.DataFrame(
    clinical_counts.items(),
    columns=['Clinical Term', 'Count']
)

clinical_df.sort_values(
    by='Count',
    ascending=False
)

In [ ]:
clinical_df_sorted = clinical_df.sort_values(
    by='Count',
    ascending=False
)

plt.figure(figsize=(10,5))

plt.bar(
    clinical_df_sorted['Clinical Term'],
    clinical_df_sorted['Count']
)

plt.xticks(rotation=45)

plt.xlabel("Clinical Term")
plt.ylabel("Frequency")
plt.title("Frequency of Common Clinical Terms")

plt.show()

### Observations

Certain findings appear much more frequently than others, showing that the dataset is not uniformly distributed across diseases.

This imbalance can affect:
- generation quality
- semantic bias
- hallucination tendencies
- model confidence

The dominance of common findings may cause the model to over-generate frequent conditions while underperforming on rarer pathologies.

### 3.4 Normal vs Abnormal Imbalance

Class imbalance is a common issue in medical datasets.

In chest X-ray reporting, normal studies are often more frequent than abnormal ones. This can lead the model to:
- over-predict normal findings
- generate generic reports
- struggle with rare abnormalities

To better understand this issue, we estimate the distribution of normal versus abnormal reports.

In [ ]:
def classify_report(report):
    report = str(report).lower()

    if "normal" in report or "no acute" in report:
        return "Normal"
    else:
        return "Abnormal"

df['report_type'] = df['report'].apply(classify_report)

report_distribution = df['report_type'].value_counts()

print(report_distribution)

In [ ]:
plt.figure(figsize=(6,5))

plt.bar(
    report_distribution.index,
    report_distribution.values
)

plt.xlabel("Report Type")
plt.ylabel("Count")
plt.title("Normal vs Abnormal Report Distribution")

plt.show()

### Observations

The dataset shows noticeable imbalance between normal and abnormal studies.

This imbalance can influence the model during training by encouraging safer or more repetitive predictions.

Understanding this distribution is important because:
- semantic evaluation becomes more meaningful
- hallucination analysis becomes more relevant
- clinically-aware evaluation becomes necessary

### 3.5 Sample Image and Report Pairs

Before training the model, it is useful to manually inspect several image-report pairs.

This helps verify:
- image quality
- report formatting
- clinical consistency
- correctness of image-report pairing

It also provides a better intuitive understanding of the complexity of the task.

In [ ]:
num_samples = 3

fig, axes = plt.subplots(1, num_samples, figsize=(15,5))

for i in range(num_samples):

    sample = df.iloc[i]

    image_path = os.path.join(
        IMAGE_DIR,
        sample['image_path']
    )

    image = Image.open(image_path).convert("RGB")

    axes[i].imshow(image)
    axes[i].set_title(f"Sample {i+1}")
    axes[i].axis("off")

    print(f"\n========== Sample {i+1} ==========")
    print("Report:")
    print(sample['report'])

### Final EDA Summary

The exploratory analysis highlights several important characteristics of the dataset:

- report lengths vary considerably
- clinical terminology follows semi-structured patterns
- certain findings dominate the dataset
- imbalance exists between normal and abnormal reports
- paired image-report samples contain diverse clinical descriptions

These observations motivate the need for:
- semantic-aware evaluation
- clinically-grounded scoring
- hallucination analysis
- careful decoder training strategies

The next section focuses on text preprocessing and vocabulary construction for the language generation pipeline.

## 4. Text Processing Pipeline

After preparing the image data, the next step is processing the radiology reports for language generation.

Unlike image classification tasks, report generation requires the model to understand and generate sequential medical text. Because of this, the reports need to be converted into a structured numerical format before they can be passed into the decoder network.

This section focuses on:
- tokenization
- vocabulary construction
- sequence encoding
- padding
- handling unknown words

These steps form the foundation of the language generation pipeline.

### 4.1 Text Cleaning and Preprocessing

Before tokenization, the reports are cleaned to improve consistency across the dataset.

The preprocessing pipeline includes:
- lowercasing text
- removing unnecessary punctuation
- removing extra spaces
- standardizing formatting

This helps reduce vocabulary fragmentation caused by small formatting differences.

In [ ]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df['clean_report'] = df['report'].apply(clean_text)

df[['report', 'clean_report']].head()

### Observations

The cleaned reports now follow a more consistent structure, which helps:
- reduce vocabulary duplication
- improve tokenizer stability
- simplify sequence encoding

At the same time, the core clinical information remains preserved.

### 4.2 Tokenization

Tokenization converts reports into smaller units called tokens.

In this project, we use word-level tokenization, where each word becomes an individual token.

For example:
- "mild cardiomegaly observed"
becomes:
- ["mild", "cardiomegaly", "observed"]

This allows the decoder to process reports as sequences of tokens during training.

In [ ]:
tokenized_reports = [
    report.split()
    for report in df['clean_report']
]

print("Sample Tokenized Report:\n")
print(tokenized_reports[0])

### 4.3 Vocabulary Construction

The vocabulary maps each token to a numerical index.

Since neural networks operate on numbers rather than raw text, every word in the dataset needs to be converted into an integer representation.

Special tokens are also added to help the decoder manage sequence generation.

### Special Tokens

The following special tokens are included:

| Token | Purpose |
|---|---|
| `<PAD>` | Padding shorter sequences |
| `<START>` | Start of report generation |
| `<END>` | End of report generation |
| `<UNK>` | Unknown or unseen words |

These tokens help standardize sequence handling during training and inference.

In [ ]:
from collections import Counter

word_counter = Counter()

for tokens in tokenized_reports:
    word_counter.update(tokens)

SPECIAL_TOKENS = ['<PAD>', '<START>', '<END>', '<UNK>']

vocab = SPECIAL_TOKENS.copy()

vocab.extend(word_counter.keys())

word2idx = {
    word: idx
    for idx, word in enumerate(vocab)
}

idx2word = {
    idx: word
    for word, idx in word2idx.items()
}

print("Vocabulary Size:", len(vocab))

In [ ]:
# example vocab mapping

sample_words = list(word2idx.items())[:10]

sample_words

### Observations

The vocabulary contains both:
- general language tokens
- clinically specific terminology

Medical datasets often contain specialized words such as:
- cardiomegaly
- atelectasis
- pneumothorax
- opacity

Because of this, handling rare or unseen terms becomes especially important for stable generation.

### 4.4 Sequence Encoding

After building the vocabulary, each token is converted into its corresponding numerical index.

The reports are also wrapped with:
- `<START>`
- `<END>`

tokens so the decoder learns where generation begins and ends.

In [ ]:
def encode_report(report_tokens):

    encoded = [word2idx['<START>']]

    for token in report_tokens:

        encoded.append(
            word2idx.get(
                token,
                word2idx['<UNK>']
            )
        )

    encoded.append(word2idx['<END>'])

    return encoded

In [ ]:
encoded_reports = [
    encode_report(tokens)
    for tokens in tokenized_reports
]

print("Encoded Sample:\n")
print(encoded_reports[0][:20])

### 4.5 Maximum Sequence Length

Radiology reports vary in length, but neural networks require fixed-length input sequences during batch training.

To handle this, a maximum sequence length is selected.

Sequences shorter than this length are padded, while extremely long sequences can be truncated if necessary.

In [ ]:
report_lengths = [
    len(report)
    for report in encoded_reports
]

MAX_SEQ_LENGTH = max(report_lengths)

print("Maximum Sequence Length:", MAX_SEQ_LENGTH)

print("Average Length:", np.mean(report_lengths))
print("Median Length:", np.median(report_lengths))

### Why Sequence Length Matters

Choosing a sequence length that is too small may remove important clinical information.

At the same time, extremely large sequence lengths:
- increase memory usage
- slow training
- add unnecessary padding

A balanced sequence length helps improve both efficiency and generation quality.

### 4.6 Sequence Padding

Since reports have different lengths, padding is used to make all sequences the same size.

Padding tokens are added to shorter reports until they match the maximum sequence length.

This allows batch training using tensors with consistent dimensions.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

padded_reports = pad_sequences(
    encoded_reports,
    maxlen=MAX_SEQ_LENGTH,
    padding='post',
    value=word2idx['<PAD>']
)

print("Padded Shape:", padded_reports.shape)

In [ ]:
# example padded sequence

print(padded_reports[0][:30])

### 4.7 Unknown Token Handling

Medical datasets can contain:
- rare terminology
- abbreviations
- uncommon findings
- spelling variations

Words that do not appear in the vocabulary are mapped to the `<UNK>` token.

This helps prevent inference failures when the model encounters unseen language during evaluation or generation.

In [ ]:
sample_word = "rareclinicalterm"

mapped_index = word2idx.get(
    sample_word,
    word2idx['<UNK>']
)

print("Mapped Index:", mapped_index)
print("Mapped Token:", idx2word[mapped_index])

### Final Text Processing Summary

At this stage:
- reports have been cleaned
- tokenization has been completed
- the vocabulary has been constructed
- reports have been encoded into numerical sequences
- sequence padding has been applied
- unknown token handling has been defined

The processed report sequences are now ready to be used by the decoder network during training.

The next section focuses on the baseline vision-language architecture used for radiology report generation.

## 5. Baseline Architecture

After preparing both the image and text pipelines, the next step is building the vision-language model used for radiology report generation.

The baseline architecture follows an encoder-attention-decoder framework, which is commonly used in image captioning and medical report generation tasks.

The overall pipeline works as follows:

1. the encoder extracts visual features from the chest X-ray
2. the attention mechanism helps the model focus on important image regions
3. the decoder generates the report word-by-word

Even though the architecture itself is relatively simple compared to large modern VLMs, it provides a strong foundation for:
- semantic analysis
- interpretability experiments
- hallucination analysis
- clinically-aware evaluation

This also allows us to focus more on semantic alignment and evaluation rather than relying only on model scale.

### Overall Architecture Pipeline

The complete pipeline can be summarized as:

```text
Chest X-ray Image
        ↓
ResNet18 Encoder
        ↓
Visual Feature Maps
        ↓
Bahdanau Attention
        ↓
Context Vector
        ↓
GRU Decoder
        ↓
Generated Radiology Report
```

The encoder extracts image representations, the attention mechanism identifies important spatial regions, and the decoder generates clinically meaningful text sequentially.

### 5.1 Encoder

The encoder is responsible for extracting visual features from the chest X-ray image.

In this project, we use a pretrained ResNet18 model as the visual backbone.

ResNet18 was selected because:
- it is lightweight and efficient
- it performs well on smaller datasets
- pretrained ImageNet weights improve feature learning
- it allows faster experimentation and interpretability analysis

Instead of training a CNN entirely from scratch, transfer learning helps the model start with useful low-level visual representations such as:
- edges
- textures
- anatomical structures

### Why ResNet18?

Although larger architectures exist, ResNet18 provides a good balance between:
- computational efficiency
- feature quality
- training stability
- interpretability

Since the Indiana dataset is relatively small, extremely large encoders may overfit easily without sufficient regularization or large-scale medical pretraining.

In [ ]:
import torchvision.models as models

resnet = models.resnet18(pretrained=True)

# Remove final classification layer
modules = list(resnet.children())[:-2]

encoder = nn.Sequential(*modules)

encoder = encoder.to(device)

print(encoder)

### Feature Extraction

The encoder converts the input image into a spatial feature map.

Instead of producing a single classification output, the final convolutional layers preserve spatial information across the image.

This is important because different image regions may correspond to:
- lungs
- heart
- pleural areas
- abnormalities

These spatial features are later used by the attention mechanism during report generation.

In [ ]:
sample_batch = torch.randn(1, 3, 224, 224).to(device)

features = encoder(sample_batch)

print("Feature Map Shape:", features.shape)

### Observations

The encoder produces high-dimensional feature maps that preserve both:
- visual semantics
- spatial structure

These feature maps act as the visual memory used by the decoder during report generation.

### 5.2 Bahdanau Attention Mechanism

Generating an entire report from a single fixed vector can limit the model’s ability to describe complex findings.

To address this, we use Bahdanau attention, also known as additive attention.

The attention mechanism allows the decoder to dynamically focus on different image regions while generating each word.

For example:
- when generating “cardiomegaly”, the model may focus more on the cardiac region
- when generating “pleural effusion”, the model may attend to lower lung regions

This improves:
- spatial grounding
- report relevance
- interpretability

### Attention Intuition

Instead of looking at the entire image equally, attention helps the model decide:

> “Which part of the image is most important for generating the next word?”

This creates a stronger connection between:
- visual regions
- generated clinical text

### Bahdanau Attention Formula

The attention score is computed as:

$$
e_{t,i} = v^T \tanh(W_h h_i + W_s s_t)
$$

Where:

- \(h_i\) represents encoder features  
- \(s_t\) represents the decoder hidden state  
- \(e_{t,i}\) is the attention score  

The attention weights are then normalized using softmax:

$$
\alpha_{t,i} =
\frac{\exp(e_{t,i})}
{\sum_j \exp(e_{t,j})}
$$

Finally, the context vector is computed as:

$$
c_t =
\sum_i \alpha_{t,i} h_i
$$

The context vector represents the weighted visual information used for generating the next token.

In [ ]:
class BahdanauAttention(nn.Module):

    def __init__(self, feature_dim, hidden_dim):

        super(BahdanauAttention, self).__init__()

        self.W_h = nn.Linear(feature_dim, hidden_dim)
        self.W_s = nn.Linear(hidden_dim, hidden_dim)

        self.V = nn.Linear(hidden_dim, 1)

    def forward(self, features, hidden_state):

        hidden_state = hidden_state.unsqueeze(1)

        score = self.V(
            torch.tanh(
                self.W_h(features) +
                self.W_s(hidden_state)
            )
        ).squeeze(-1)

        attention_weights = torch.softmax(score, dim=1)

        context_vector = torch.sum(
            attention_weights.unsqueeze(-1) * features,
            dim=1
        )

        return context_vector, attention_weights

### Why Attention Matters

Without attention, the decoder would rely on a single compressed image representation.

Attention improves generation by allowing the model to:
- revisit relevant image regions
- dynamically shift focus
- preserve spatial context during decoding

Attention maps also help us later during interpretability analysis by visualizing where the model focuses while generating different clinical terms.

### 5.3 GRU Decoder

The decoder is responsible for generating the radiology report one token at a time.

In this project, we use a GRU-based decoder.

GRUs are recurrent neural networks designed to model sequential dependencies while being computationally lighter than LSTMs.

The decoder receives:
- the previous generated token
- the context vector from attention
- the previous hidden state

and predicts the next word in the sequence.

### Why GRU?

We selected GRUs because they:
- train efficiently on smaller datasets
- handle sequential text generation well
- require fewer parameters than LSTMs
- reduce computational complexity

This makes them suitable for:
- lightweight experimentation
- interpretability-focused analysis
- limited medical datasets

### GRU Update Equations

The GRU updates its hidden state using gating mechanisms.

Update gate:

$$
z_t =
\sigma(W_z x_t + U_z h_{t-1})
$$

Reset gate:

$$
r_t =
\sigma(W_r x_t + U_r h_{t-1})
$$

Candidate hidden state:

$$
\tilde{h}_t =
\tanh(
W_h x_t +
U_h(r_t \odot h_{t-1})
)
$$

Final hidden state:

$$
h_t =
(1-z_t)\odot h_{t-1}
+
z_t \odot \tilde{h}_t
$$

These gates help preserve long-range contextual information during report generation.

In [ ]:
class DecoderGRU(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        feature_dim
    ):

        super(DecoderGRU, self).__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.gru = nn.GRU(
            embedding_dim + feature_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(
        self,
        word_input,
        context_vector,
        hidden_state
    ):

        embedded = self.embedding(word_input)

        gru_input = torch.cat(
            [embedded, context_vector.unsqueeze(1)],
            dim=-1
        )

        output, hidden_state = self.gru(
            gru_input,
            hidden_state
        )

        output = self.fc(output)

        return output, hidden_state

### 5.4 Full Encoder-Attention-Decoder Pipeline

The final baseline architecture combines:
- ResNet18 encoder
- Bahdanau attention
- GRU decoder

During training:
1. the encoder extracts image features
2. attention computes spatial importance
3. the decoder generates reports sequentially

This architecture forms the baseline pipeline used throughout the project before introducing:
- semantic-aware losses
- clinically-aware evaluation
- interpretability analysis
- hallucination studies

### Architecture Diagram

```text
Input Chest X-ray
        ↓
ResNet18 Encoder
        ↓
Spatial Feature Maps
        ↓
Bahdanau Attention
        ↓
Context Vector
        ↓
GRU Decoder
        ↓
Generated Report Tokens
```

This pipeline allows the model to connect:
- image regions
- semantic representations
- clinical language generation
within a unified vision-language framework.

### Baseline Architecture Summary

The baseline model provides:
- efficient image feature extraction
- spatial attention-based grounding
- sequential clinical text generation

Although the architecture itself is relatively lightweight, it creates a strong foundation for:
- semantic-aware evaluation
- hallucination analysis
- clinically-aware metrics
- interpretability experiments

The next section focuses on the proposed improvements built on top of this baseline framework.

## 6. Proposed Scarlet Architecture

SCARLET stands for:

**Semantic Clinical Alignment for Radiology Language Evaluation and Training**

The baseline model gives us a working encoder-attention-decoder pipeline. However, for the final model, we want to move beyond simple report generation and focus more directly on semantic and clinical correctness.

SCARLET is designed around this idea:

> A generated radiology report should not only sound fluent. It should preserve the correct clinical meaning.

So instead of only optimizing for word prediction, SCARLET combines:
- a stronger visual encoder
- attention-based report generation
- semantic-aware training
- clinically-aware evaluation
- hallucination analysis

The goal is to make the model more aligned with medical meaning, not just exact wording.

### 6.1 SCARLET Pipeline Overview

The proposed SCARLET framework follows this structure:

```text
Chest X-ray Image
        ↓
DenseNet121 Encoder
        ↓
Visual Feature Maps
        ↓
Bahdanau Attention
        ↓
GRU Decoder
        ↓
Generated Radiology Report
        ↓
SCARLET Score + Clinical Consistency Analysis

During training, SCARLET also introduces a semantic-aware loss term so that the model is encouraged to generate reports that are closer to the ground-truth report in meaning, not only in token sequence.

### 6.2 DenseNet121 Visual Encoder

For the proposed model, we use DenseNet121 as the visual encoder.

DenseNet121 is commonly used in medical imaging because it encourages feature reuse across layers. Each layer receives information from previous layers, which helps preserve useful low-level and high-level visual patterns.

This is useful for chest X-rays because important findings can be subtle, such as:
- small opacities
- enlarged heart silhouette
- pleural changes
- lung texture abnormalities

Compared with ResNet18, DenseNet121 gives a stronger visual backbone while still being practical to train.

In [ ]:
from torchvision import models
import torch.nn as nn

densenet = models.densenet121(pretrained=True)

# remove classifier head
encoder = densenet.features

encoder = encoder.to(device)

print(encoder)

### 6.3 Feature Map Extraction

DenseNet121 produces spatial feature maps rather than a single classification label.

This matters because radiology report generation needs spatial information. The decoder should be able to refer to different regions of the image when generating different clinical terms.

For example:
- heart-related terms should depend more on cardiac-region features
- lung opacity terms should depend more on lung-field features
- pleural findings should depend more on boundary-region features

These feature maps are passed into the attention mechanism.

In [ ]:
sample_batch = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    feature_maps = encoder(sample_batch)

print("DenseNet121 Feature Map Shape:", feature_maps.shape)

### 6.4 Flattening Feature Maps for Attention

The attention mechanism expects a sequence of visual feature vectors.

So, we reshape the DenseNet feature map from:

$$
B \times C \times H \times W
$$

to:

$$
B \times (H \cdot W) \times C
$$

Where:
- \(B\) is the batch size
- \(C\) is the feature dimension
- \(H\) and \(W\) are spatial dimensions

This allows the decoder to attend over different spatial locations in the image.

In [ ]:
def flatten_feature_maps(feature_maps):
    """
    Converts feature maps from:
    [batch_size, channels, height, width]
    to:
    [batch_size, num_locations, channels]
    """

    batch_size, channels, height, width = feature_maps.size()

    features = feature_maps.view(
        batch_size,
        channels,
        height * width
    )

    features = features.permute(0, 2, 1)

    return features

In [ ]:
flattened_features = flatten_feature_maps(feature_maps)

print("Flattened Feature Shape:", flattened_features.shape)

### 6.5 Attention-Based Visual Grounding

SCARLET keeps the attention mechanism because it gives the decoder access to spatial visual information while generating each token.

The attention mechanism helps answer:

> Which image region is most relevant for the next generated word?

This is important for interpretability because we can later visualize attention maps and compare them with generated clinical terms.

We do not claim that attention is a perfect explanation. Instead, we use it as one supporting signal alongside quantitative embedding and clinical consistency analysis.

### 6.6 GRU Decoder for Report Generation

The GRU decoder generates the radiology report one token at a time.

At each time step, the decoder receives:
- the previous word embedding
- the attention context vector
- the previous hidden state

It then predicts the next token in the report.

We keep the GRU decoder because it is:
- efficient
- stable on smaller datasets
- easier to interpret than a large decoder
- suitable for controlled ablation studies

This allows the main contribution to stay focused on semantic-clinical alignment rather than model size alone.

### 6.7 Semantic Clinical Alignment Loss

Traditional cross-entropy loss teaches the decoder to predict the next token.

However, radiology reports can be medically equivalent even when their wording differs.

For example:

```text
Ground truth: enlarged cardiac silhouette
Prediction: cardiomegaly
```

These phrases are semantically close, but token-level loss may still penalize them heavily.

To address this, SCARLET adds a semantic alignment term to the training objective.

### Cross-Entropy Loss

The standard token-level loss is:

$$
L_{CE}
=
-\sum_{t=1}^{T}
y_t \log(\hat{y}_t)
$$

This encourages the model to generate the correct next token at each time step.

### Coverage Loss

Coverage loss is used to reduce repetitive attention patterns.

$$
L_{cov}
=
\sum_t \sum_i
\min(a_i^t, c_i^t)
$$

Where:
- \(a_i^t\) is the attention weight at location \(i\) and time step \(t\)
- \(c_i^t\) is the accumulated attention over previous time steps

This helps discourage the decoder from repeatedly attending to the same image regions.

### Semantic Alignment Loss

We define semantic alignment loss as:

$$
L_{sem}
=
1 -
\cos(E(y_{pred}), E(y_{true}))
$$

Where:
- \(E(y_{pred})\) is the sentence embedding of the generated report
- \(E(y_{true})\) is the sentence embedding of the ground-truth report
- cosine similarity measures semantic closeness

This loss encourages generated reports to stay closer to the ground truth in meaning.

### Final SCARLET Training Objective

The final training loss is:

$$
L_{SCARLET}
=
L_{CE}
+
\lambda L_{cov}
+
\mu L_{sem}
$$

Where:
- \(\lambda\) controls the coverage penalty
- \(\mu\) controls the semantic alignment penalty

This gives the model three learning signals:
- token correctness
- reduced repetition
- semantic similarity

### 6.8 SCARLET Score

Alongside the training objective, we introduce the SCARLET Score for evaluation.

The SCARLET Score combines:
- semantic similarity
- lexical overlap
- clinical concept alignment

The metric is defined as:

$$
S_{SCARLET}
=
\alpha S_{sem}
+
\beta S_{lex}
+
\gamma S_{clin}
$$

Where:
- \(S_{sem}\) is semantic similarity, such as BERTScore
- \(S_{lex}\) is lexical overlap, such as ROUGE-L
- \(S_{clin}\) is clinical concept alignment

With:

$$
\alpha + \beta + \gamma = 1
$$

This keeps the metric interpretable while making it more clinically aware than lexical metrics alone.

### 6.9 Clinical Concept Alignment

The clinical component checks whether important medical concepts are preserved between the generated and ground-truth reports.

We define:

$$
S_{clin}
=
\frac{|C_{pred} \cap C_{true}|}
{|C_{true}| + \epsilon}
$$

Where:
- \(C_{pred}\) is the set of clinical concepts extracted from the generated report
- \(C_{true}\) is the set of clinical concepts extracted from the ground-truth report
- \(\epsilon\) prevents division by zero

This rewards reports that preserve clinically important findings, even when the wording is different.

### 6.10 Example Clinical Synonym Mapping

A lightweight synonym map is used to normalize common clinical terms.

| Original Term | Normalized Concept |
|---|---|
| cardiomegaly | enlarged heart |
| enlarged cardiac silhouette | enlarged heart |
| opacity | lung opacity |
| infiltrate | lung opacity |
| pleural fluid | pleural effusion |
| no acute disease | normal |
| no acute cardiopulmonary abnormality | normal |

This does not replace expert clinical evaluation, but it gives us a practical way to test whether the model preserves important medical meaning.

In [ ]:
clinical_synonyms = {
    "cardiomegaly": "enlarged_heart",
    "enlarged heart": "enlarged_heart",
    "enlarged cardiac silhouette": "enlarged_heart",
    "opacity": "lung_opacity",
    "opacities": "lung_opacity",
    "infiltrate": "lung_opacity",
    "infiltration": "lung_opacity",
    "pleural fluid": "pleural_effusion",
    "pleural effusion": "pleural_effusion",
    "effusion": "pleural_effusion",
    "pneumothorax": "pneumothorax",
    "atelectasis": "atelectasis",
    "edema": "edema",
    "no acute disease": "normal",
    "no acute cardiopulmonary abnormality": "normal",
    "normal": "normal"
}


def extract_clinical_concepts(report, synonym_map):
    report = str(report).lower()

    concepts = set()

    for term, concept in synonym_map.items():
        if term in report:
            concepts.add(concept)

    return concepts

In [ ]:
def clinical_alignment_score(pred_report, true_report, synonym_map, epsilon=1e-8):

    pred_concepts = extract_clinical_concepts(pred_report, synonym_map)
    true_concepts = extract_clinical_concepts(true_report, synonym_map)

    if len(true_concepts) == 0:
        return 1.0 if len(pred_concepts) == 0 else 0.0

    overlap = len(pred_concepts.intersection(true_concepts))

    score = overlap / (len(true_concepts) + epsilon)

    return score

### 6.11 Final SCARLET Framework Summary

SCARLET combines generation, training, and evaluation into one semantic-clinical framework.

The final framework includes:

1. DenseNet121 encoder for stronger visual feature extraction
2. Bahdanau attention for spatial grounding
3. GRU decoder for report generation
4. coverage loss to reduce repetition
5. semantic alignment loss to improve meaning preservation
6. SCARLET Score for clinically-aware evaluation
7. hallucination and interpretability analysis for deeper model understanding

This keeps the architecture practical while making the research contribution more meaningful.

### 6.12 Why SCARLET Is Different From the Baseline

The baseline model mainly asks:

> Can we generate a report from an X-ray?

SCARLET asks a more clinically relevant question:

> Does the generated report preserve the correct medical meaning?

This is the main shift.

Instead of only reporting BLEU or ROUGE, SCARLET evaluates whether generated reports are semantically and clinically aligned with the reference report.